In [ ]:
import gzip
import json
import pandas as pd

def stream_json_gz(path, domain_name, max_items=None):
    data = []
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_items and i >= max_items:
                break
            record = json.loads(line)
            record["domain"] = domain_name
            data.append(record)
    return pd.DataFrame(data)


electronics = stream_json_gz(
    "/DATA/shourya_2211mc14/Sougata2/mmprodnet/datasets/meta_Electronics.jsonl.gz",
    domain_name="electronics",
    max_items=200000
)

clothing = stream_json_gz(
    "/DATA/shourya_2211mc14/Sougata2/mmprodnet/datasets/meta_Clothing_Shoes_and_Jewelry.jsonl.gz",
    domain_name="clothing",
    max_items=200000
)

home = stream_json_gz(
    "/DATA/shourya_2211mc14/Sougata2/mmprodnet/datasets/meta_Home_and_Kitchen.jsonl.gz",
    domain_name="home",
    max_items=200000
)

df = pd.concat([electronics, clothing, home], ignore_index=True)

print("Total rows:", len(df))
print("Columns:", df.columns.tolist())
df.head(2)

In [ ]:
print("Total rows:", len(df))

print("\nMissing Ratio:")
print(df.isnull().mean().sort_values(ascending=False))

print("\nUnique parent_asin count:", df["parent_asin"].nunique())

print("\nExample categories:")
print(df["categories"].iloc[0])

print("\nExample details:")
print(df["details"].iloc[0])

In [ ]:
# ==================================
# Find maximum category depth
# ==================================

# compute length of category lists
df["category_depth"] = df["categories"].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

print("Maximum depth:", df["category_depth"].max())

print("\nDepth distribution:")
print(df["category_depth"].value_counts().sort_index())

print("\nExample rows with maximum depth:")
max_depth = df["category_depth"].max()
print(df[df["category_depth"] == max_depth]["categories"].head(10))

In [ ]:
# ==================================
# Standardize dataframe for graph engineering
# ==================================

# rename parent_asin -> item_id
df = df.rename(columns={"parent_asin": "item_id"})

# keep only important columns for KG construction
columns_to_keep = [
    "item_id",
    "title",
    "categories",
    "details",
    "store",
    "main_category",
    "average_rating",
    "rating_number",
    "features",
    "description",
    "images",
    "domain"
]

df = df[columns_to_keep]

# remove rows without id or title
df = df.dropna(subset=["item_id", "title"])

# reset index
df = df.reset_index(drop=True)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nSample row:")
print(df.iloc[0])

In [ ]:
# ==================================
# Build taxonomy edges
# ==================================

taxonomy_edges = []
item_category_edges = []

for _, row in df.iterrows():

    cats = row["categories"]
    item = row["item_id"]

    if not isinstance(cats, list) or len(cats) == 0:
        continue

    # item → leaf category
    leaf = cats[-1]
    item_category_edges.append((f"item_{item}", "belongs_to", leaf))

    # category hierarchy
    for i in range(1, len(cats)):
        parent = cats[i-1]
        child = cats[i]

        taxonomy_edges.append((child, "parent_category", parent))


taxonomy_edges = pd.DataFrame(
    taxonomy_edges,
    columns=["head", "relation", "tail"]
)

item_category_edges = pd.DataFrame(
    item_category_edges,
    columns=["head", "relation", "tail"]
)

print("Taxonomy edges:", len(taxonomy_edges))
print("Item-category edges:", len(item_category_edges))

taxonomy_edges.head()

In [ ]:
item_category_edges.head()

In [ ]:
# ==================================
# Clean taxonomy edges
# ==================================

# remove duplicate edges
taxonomy_edges = taxonomy_edges.drop_duplicates()

# also remove self loops if any
taxonomy_edges = taxonomy_edges[
    taxonomy_edges["head"] != taxonomy_edges["tail"]
]

print("Clean taxonomy edges:", len(taxonomy_edges))

print("\nExample taxonomy edges:")
print(taxonomy_edges.head())

In [ ]:
# ==================================
# Save taxonomy graph artifacts
# ==================================

taxonomy_edges.to_parquet("taxonomy_edges.parquet", index=False)
item_category_edges.to_parquet("item_category_edges.parquet", index=False)

print("Saved:")
print("taxonomy_edges.parquet")
print("item_category_edges.parquet")

In [ ]:
import re

def normalize_text(x):
    if not isinstance(x, str):
        return None
    
    x = x.strip().lower()
    x = re.sub(r"\s+", " ", x)
    
    if len(x) < 2:
        return None
    if x.isdigit():
        return None
    
    return x


def extract_clean_brand(details_dict):
    if not isinstance(details_dict, dict):
        return None
    
    # Priority 1: Brand
    for key, value in details_dict.items():
        if key.lower() == "brand":
            return normalize_text(value)
    
    # Priority 2: Manufacturer
    for key, value in details_dict.items():
        if "manufacturer" in key.lower():
            return normalize_text(value)
    
    return None


# Extract brand
df["brand"] = df["details"].apply(extract_clean_brand)
df["seller"] = df["store"].apply(normalize_text)




print("Brand missing ratio:", df["brand"].isnull().mean())
print("Seller missing ratio:", df["seller"].isnull().mean())

print("\nUnique brands:", df["brand"].nunique())
print("Unique sellers:", df["seller"].nunique())

print("\nTop 10 brands:")
print(df["brand"].value_counts().head(10))

print("\nTop 10 sellers:")
print(df["seller"].value_counts().head(10))

In [ ]:
# ==================================
# Remove noisy brand nodes
# ==================================

bad_brands = {
    "generic",
    "no",
    "none",
    "unknown",
    "n/a",
    "na",
    "unbranded"
}

df.loc[df["brand"].isin(bad_brands), "brand"] = None

print("Brand missing ratio after cleaning:", df["brand"].isnull().mean())

print("\nUnique brands after cleaning:", df["brand"].nunique())

print("\nTop brands after cleaning:")
print(df["brand"].value_counts().head(10))

In [ ]:
# ================================
# Brand distribution analysis
# ================================

brand_counts = df["brand"].value_counts()

print("Total unique brands:", len(brand_counts))

print("\nTop 20 brands:")
print(brand_counts.head(20))

print("\nBrands with only 1 product:", (brand_counts == 1).sum())

print("\nBrands with <5 products:", (brand_counts < 5).sum())

In [ ]:
# ==================================
# Filter rare brands (frequency < 3)
# ==================================

brand_counts = df["brand"].value_counts()

valid_brands = brand_counts[brand_counts >= 5].index

df["brand"] = df["brand"].where(df["brand"].isin(valid_brands))

print("Remaining unique brands:", df["brand"].nunique())
print("Brand missing ratio after filtering:", df["brand"].isnull().mean())

In [ ]:
# ===============================
# Build item -> brand edges
# ===============================

item_brand_edges = df.dropna(subset=["brand"])[["item_id", "brand"]].copy()

# FIX (ghost-node bug): prefix item head with "item_" to match
# attribute / item-item naming. Without this, has_brand attaches to a
# bare-ASIN node that has no text/image and no other edges.
item_brand_edges["head"] = "item_" + item_brand_edges["item_id"].astype(str)
item_brand_edges["tail"] = item_brand_edges["brand"]
item_brand_edges["relation"] = "has_brand"

item_brand_edges = item_brand_edges[["head", "relation", "tail"]]

print("Item -> Brand edges:", len(item_brand_edges))

item_brand_edges.head()


In [ ]:
print("Average items per brand:",
      item_brand_edges.groupby("tail").size().mean())

print("Max items in a brand:",
      item_brand_edges.groupby("tail").size().max())

In [ ]:
item_brand_edges.to_parquet("item_brand_edges_multimodal.parquet", index=False)

In [ ]:
# ==================================
# Robust two-level blocking (leaf + parent)
# ==================================

block_leaf = {}
block_parent = {}

for _, row in df.iterrows():

    item = row["item_id"]
    cats = row["categories"]

    if not isinstance(cats, list) or len(cats) == 0:
        continue

    # leaf category block
    leaf = cats[-1]
    block_leaf.setdefault(leaf, []).append(item)

    # parent category block
    if len(cats) > 1:
        parent = cats[-2]
        block_parent.setdefault(parent, []).append(item)


print("Leaf blocks:", len(block_leaf))
print("Parent blocks:", len(block_parent))

print("\nExample leaf block:")
for k,v in list(block_leaf.items())[:1]:
    print(k, "→", len(v), "items")

print("\nExample parent block:")
for k,v in list(block_parent.items())[:1]:
    print(k, "→", len(v), "items")

In [ ]:
block_leaf

In [ ]:
block_parent

In [ ]:
for i in range(20,30):
    print(df['title'].iloc[i])

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
# Create leaf_category column

def get_leaf(cat):
    if isinstance(cat, list) and len(cat) > 0:
        return cat[-1]
    return None

df["leaf_category"] = df["categories"].apply(get_leaf)

print("Unique leaf categories:", df["leaf_category"].nunique())
print(df["leaf_category"].value_counts().head(10))

In [ ]:
df[["categories","leaf_category"]].sample(10)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

titles = df["title"].fillna("").tolist()

embeddings = model.encode(
    titles,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)

In [ ]:
np.save("title_embeddings.npy", embeddings)

In [ ]:
import numpy as np

leaf_sizes = [len(v) for v in block_leaf.values()]

print("Leaf blocks:", len(leaf_sizes))
print("Average size:", np.mean(leaf_sizes))
print("Median size:", np.median(leaf_sizes))
print("Max size:", np.max(leaf_sizes))
print("Min size:", np.min(leaf_sizes))

In [ ]:
parent_sizes = [len(v) for v in block_parent.values()]

print("Parent blocks:", len(parent_sizes))
print("Average size:", np.mean(parent_sizes))
print("Median size:", np.median(parent_sizes))
print("Max size:", np.max(parent_sizes))
print("Min size:", np.min(parent_sizes))

In [ ]:
sorted_leaf = sorted(block_leaf.items(), key=lambda x: len(x[1]), reverse=True)

for k,v in sorted_leaf[:10]:
    print(k, len(v))

In [ ]:
sorted_parent = sorted(block_parent.items(), key=lambda x: len(x[1]), reverse=True)

for k,v in sorted_parent[:10]:
    print(k, len(v))

In [ ]:
example = list(block_leaf.keys())[3]

print("BLOCK:", example)

items = block_leaf[example][:10]

df[df["item_id"].isin(items)][["item_id","title"]]

In [ ]:
import matplotlib.pyplot as plt

plt.hist(leaf_sizes, bins=50)
plt.title("Leaf Block Size Distribution")
plt.xlabel("Block size")
plt.ylabel("Count")
plt.show()

In [ ]:
for name, items in sorted_leaf[:10]:
    print(name, len(items))

In [ ]:
largest_leaf = sorted_leaf[3][0]
largest_items = block_leaf[largest_leaf]

print("Block:", largest_leaf)
print("Size:", len(largest_items))

df[df["item_id"].isin(largest_items)][["title"]].head(20)

In [ ]:
df[df["item_id"].isin(largest_items)]["brand"].value_counts().head(10)

In [ ]:
embeddings = np.load("title_embeddings.npy")

In [ ]:
embeddings.shape

In [ ]:
block_name = list(block_leaf.keys())[0]

print("Block name:", block_name)

print("First 10 elements in block:")
print(block_leaf[block_name][:10])

In [ ]:
id_to_index = {item_id: idx for idx, item_id in enumerate(df["item_id"])}

print("Mapping size:", len(id_to_index))

In [ ]:
def convert_items_to_indices(items):
    
    return [id_to_index[i] for i in items if i in id_to_index]

In [ ]:
import faiss

K = 20
candidate_pairs = []

def process_blocks(block_dict, block_type):

    for block_name, items in block_dict.items():

        indices = convert_items_to_indices(items)

        if len(indices) < 2:
            continue

        block_vectors = embeddings[indices]

        dim = block_vectors.shape[1]

        index = faiss.IndexFlatIP(dim)
        index.add(block_vectors)

        scores, neighbors = index.search(block_vectors, K+1)

        for i, item_idx in enumerate(indices):

            for j in range(1, K+1):

                neighbor_pos = neighbors[i][j]

                neighbor_item = indices[neighbor_pos]

                candidate_pairs.append(
                    (
                        item_idx,
                        neighbor_item,
                        scores[i][j],
                        block_name,
                        block_type
                    )
                )

In [ ]:
process_blocks(block_leaf, "leaf")

process_blocks(block_parent, "parent")

In [ ]:
candidates = pd.DataFrame(
    candidate_pairs,
    columns=[
        "item_i",
        "item_j",
        "similarity",
        "block",
        "block_type"
    ]
)

candidates["pair_key"] = candidates.apply(
    lambda x: tuple(sorted((x.item_i, x.item_j))),
    axis=1
)

candidates = (
    candidates
    .sort_values("similarity", ascending=False)
    .drop_duplicates("pair_key")
    .reset_index(drop=True)
)

In [ ]:
# FIX: do NOT rebuild `candidates` here.
# The cell above already deduplicated on pair_key. Rebuilding from
# candidate_pairs would silently undo the dedup (both (A,B) and (B,A),
# plus the same pair from leaf and parent blocks, would return).

print("Deduplicated candidate pairs:", len(candidates))
candidates.head()


In [ ]:
candidates.to_parquet("candidate_pairs.parquet", index=False)

In [ ]:
import ast

def parse_details(detail):
    # Already dictionary
    if isinstance(detail, dict):
        return detail
    
    # If string representation of dict
    try:
        return ast.literal_eval(detail)
    except:
        return {}

df["parsed_details"] = df["details"].apply(parse_details)

print("Parsing done.")
print("Example parsed_details:")
print(df["parsed_details"].iloc[0])
print("Type:", type(df["parsed_details"].iloc[0]))

avg_attr = df["parsed_details"].apply(len).mean()
print("Average number of attributes per item:", avg_attr)

In [ ]:
from collections import Counter

attribute_counter = Counter()

for d in df["parsed_details"]:
    for key, val in d.items():
        attribute_counter[key] += 1

print("Top 20 most frequent attribute keys:")
for k, v in attribute_counter.most_common(20):
    print(k, ":", v)

print("\nTotal unique attribute keys:", len(attribute_counter))

In [ ]:
IDENTITY_KEYS = [
    "Color",
    "Size",
    "Material",
    "Style",
    "Pattern",
    "Shape",
    "Compatible Devices",
    "Special Feature"
]

def extract_identity_attributes(detail_dict):
    filtered = {}
    for k, v in detail_dict.items():
        if k in IDENTITY_KEYS:
            filtered[k] = v
    return filtered

df["identity_attributes"] = df["parsed_details"].apply(extract_identity_attributes)

print("Done extracting identity attributes.")

In [ ]:
from collections import Counter

attribute_value_counter = Counter()

# Normalize and count attribute values
for attrs in df["identity_attributes"]:
    for k, v in attrs.items():
        if isinstance(v, str):
            # split multi-values like "Blue,White"
            values = [x.strip().lower() for x in v.split(",")]
            for val in values:
                node = f"{k}:{val}"
                attribute_value_counter[node] += 1

print("Total attribute-value pairs:", len(attribute_value_counter))

print("\nTop 20 attribute-value pairs:")
for k, v in attribute_value_counter.most_common(20):
    print(k, ":", v)

In [ ]:
rare_count = sum(1 for k, v in attribute_value_counter.items() if v == 1)
freq_2_5 = sum(1 for k, v in attribute_value_counter.items() if 2 <= v <= 5)

print("Attribute-values appearing once:", rare_count)
print("Attribute-values appearing 2-5 times:", freq_2_5)

In [ ]:
for threshold in [5, 10, 20, 50]:
    kept = sum(1 for k, v in attribute_value_counter.items() if v >= threshold)
    print(f"Threshold ≥{threshold}: {kept} nodes")

In [ ]:
MIN_FREQ = 10

valid_attribute_values = {
    attr for attr, count in attribute_value_counter.items()
    if count >= MIN_FREQ
}

print("Attribute nodes after pruning:", len(valid_attribute_values))


def normalize_key(k):
    return k.lower().replace(" ", "_")


item_attribute_triplets = []

for item_id, attrs in zip(df["item_id"], df["identity_attributes"]):

    for k, v in attrs.items():

        if not isinstance(v, str):
            continue

        key = normalize_key(k)

        values = [x.strip().lower() for x in v.split(",")]

        for val in values:

            node = f"{k}:{val}"

            if node in valid_attribute_values:

                relation = f"has_{key}"

                attribute_node = f"attr_{key}_{val}"

                item_attribute_triplets.append(
                    (f"item_{item_id}", relation, attribute_node)
                )


print("Total attribute triplets:", len(item_attribute_triplets))
print("Unique attribute nodes:", len(set([t[2] for t in item_attribute_triplets])))
print("Items covered:", len(set([t[0] for t in item_attribute_triplets])))

In [ ]:
import pandas as pd

attr_edges = pd.DataFrame(
    item_attribute_triplets,
    columns=["head","relation","tail"]
)

attr_edges.to_parquet("item_attribute_edges.parquet", index=False)

print("Saved attribute edges.")

In [ ]:
attr_edges.head()

In [ ]:
item_attr_map = {}

for item, relation, attr in item_attribute_triplets:
    
    if item not in item_attr_map:
        item_attr_map[item] = set()
        
    item_attr_map[item].add(attr)

In [ ]:
def attribute_overlap(item_i, item_j):
    
    attrs_i = item_attr_map.get(item_i, set())
    attrs_j = item_attr_map.get(item_j, set())
    
    if len(attrs_i | attrs_j) == 0:
        return 0
        
    return len(attrs_i & attrs_j) / len(attrs_i | attrs_j)

In [ ]:
candidates["attr_overlap"] = candidates.apply(
    lambda x: attribute_overlap(
        f"item_{df.loc[x['item_i'],'item_id']}",
        f"item_{df.loc[x['item_j'],'item_id']}"
    ),
    axis=1
)

In [ ]:
import pandas as pd

# FIX: None == None must NOT count as a brand match (~40% of items lack a brand).
_bi = df.loc[candidates["item_i"], "brand"].values
_bj = df.loc[candidates["item_j"], "brand"].values
candidates["brand_match"] = (
    (_bi == _bj) & pd.notna(_bi) & pd.notna(_bj)
).astype(int)


In [ ]:
import pandas as pd

# FIX: missing-vs-missing leaf must NOT count as a category match.
_ci = df.loc[candidates["item_i"], "leaf_category"].values
_cj = df.loc[candidates["item_j"], "leaf_category"].values
candidates["category_match"] = (
    (_ci == _cj) & pd.notna(_ci) & pd.notna(_cj)
).astype(int)


In [ ]:
candidates['brand_match'].value_counts()

In [ ]:
candidates[[
    "similarity",
    "attr_overlap",
    "brand_match",
    "category_match"
]].describe()

In [ ]:
import numpy as np

candidates = candidates.replace([np.inf, -np.inf], np.nan)

candidates = candidates.dropna(subset=["similarity"])

In [ ]:
candidates = candidates[candidates["similarity"] > -1]

In [ ]:
candidates["similarity"].describe()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(candidates["similarity"], bins=50)
plt.title("Text Similarity Distribution")
plt.xlabel("similarity")
plt.ylabel("count")
plt.show()

In [ ]:
plt.hist(candidates["attr_overlap"], bins=50)
plt.title("Attribute Overlap Distribution")
plt.show()

In [ ]:
top_pairs = candidates.sort_values(
    "similarity", ascending=False
).head(20)

top_pairs[["item_i","item_j","similarity","attr_overlap"]]

In [ ]:
for _, r in top_pairs.iterrows():
    print(df.loc[r["item_i"], "title"])
    print(df.loc[r["item_j"], "title"])
    print("---")

In [ ]:
low_pairs = candidates.sort_values(
    "similarity"
).head(20)

In [ ]:
for _, r in low_pairs.iterrows():
    print(df.loc[r["item_i"], "title"])
    print(df.loc[r["item_j"], "title"])
    print("---")

In [ ]:
candidates[[
    "similarity",
    "attr_overlap",
    "brand_match",
    "category_match"
]].corr()

In [ ]:
candidates["block_type"].value_counts()

In [ ]:
len(candidates)

In [ ]:
len(set(candidates["item_i"]))

In [ ]:
candidates.to_parquet("candidate_feature_table.parquet", index=False)

In [ ]:
import pickle

with open("item_attribute_map.pkl", "wb") as f:
    pickle.dump(item_attr_map, f)

In [ ]:
with open("leaf_blocks.pkl","wb") as f:
    pickle.dump(block_leaf,f)

with open("parent_blocks.pkl","wb") as f:
    pickle.dump(block_parent,f)

In [ ]:
print(df["images"].iloc[0])
print(type(df["images"].iloc[0]))

In [ ]:
def extract_image_url(img_list):

    if not isinstance(img_list, list) or len(img_list) == 0:
        return None

    img = img_list[0]

    if isinstance(img, dict):

        if img.get("hi_res"):
            return img["hi_res"]

        if img.get("large"):
            return img["large"]

        if img.get("thumb"):
            return img["thumb"]

    return None


df["image_url"] = df["images"].apply(extract_image_url)

print("Image coverage:", df["image_url"].notnull().mean())

df[["item_id","image_url"]].head()

In [ ]:
df[["item_id","image_url"]].to_parquet(
    "item_image_urls.parquet",
    index=False
)

In [ ]:
# items appearing in candidate pairs
candidate_items = set(candidates["item_i"]).union(
    set(candidates["item_j"])
)

print("Candidate items:", len(candidate_items))

# convert set → list for pandas indexing
candidate_items = list(candidate_items)

df_images = df.loc[candidate_items, ["item_id", "image_url"]]

print("Rows for image embedding:", len(df_images))

df_images.head()

In [ ]:
import requests
from PIL import Image
from io import BytesIO

def load_image(url):
    try:
        response = requests.get(url, timeout=5)
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return image
    except:
        return None


# test on a few images
for url in df_images["image_url"].head(5):
    img = load_image(url)
    print(img)

In [ ]:

from transformers import CLIPProcessor, CLIPModel
import torch

# choose GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# load model
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("CLIP model loaded.")

In [ ]:
# test CLIP embedding on a few images
test_urls = df_images["image_url"].head(5)

for url in test_urls:

    img = load_image(url)

    if img is None:
        print("Image failed to load")
        continue

    inputs = processor(images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        features = model.get_image_features(**inputs)

    # Add .pooler_output before .cpu()
    emb = features.pooler_output.cpu().numpy()[0]

    print("Embedding shape:", emb.shape)

    print("Embedding shape:", emb.shape)

In [ ]:
import numpy as np
import concurrent.futures
from tqdm import tqdm

BATCH_SIZE = 32
# This controls how many downloads happen at the exact same time. 
# 16 or 32 is usually a great sweet spot for network requests.
MAX_WORKERS = 16 

image_embeddings = []
image_item_ids = []

rows = df_images.reset_index(drop=True)

# 1. Create a quick helper function for the thread pool
def fetch_image(row):
    """Downloads an image and returns it paired with its ID."""
    img = load_image(row["image_url"])
    return row["item_id"], img

# 2. Start the main loop
for start in tqdm(range(0, len(rows), BATCH_SIZE)):

    batch = rows.iloc[start:start+BATCH_SIZE]
    
    # Convert batch to a list of dictionaries for easier mapping
    row_dicts = batch.to_dict("records")
    
    images = []
    ids = []

    # 3. Download the entire batch concurrently!
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # executor.map runs 'fetch_image' on every row in 'row_dicts' simultaneously
        results = list(executor.map(fetch_image, row_dicts))
        
    # 4. Unpack results and filter out failed downloads
    for item_id, img in results:
        if img is not None:
            images.append(img)
            ids.append(item_id)

    if len(images) == 0:
        continue

    # 5. Run the downloaded images through CLIP
    inputs = processor(images=images, return_tensors="pt").to(device)

    with torch.no_grad():
        features = model.get_image_features(**inputs)

    # Remember our fix from earlier!
    emb = features.pooler_output.cpu().numpy()

    image_embeddings.append(emb)
    image_item_ids.extend(ids)

# Combine batches
image_embeddings = np.vstack(image_embeddings)

print("Final Embeddings shape:", image_embeddings.shape)
print(f"Total successful items: {len(image_item_ids)}")

In [ ]:
np.save("image_embeddings.npy", image_embeddings)
np.save("image_item_ids.npy", image_item_ids)

In [ ]:
import numpy as np

# load embeddings
image_embeddings = np.load("image_embeddings.npy")
image_item_ids = np.load("image_item_ids.npy", allow_pickle=True)

print("Embeddings shape:", image_embeddings.shape)

# build lookup dictionary
image_emb_map = {
    item_id: emb for item_id, emb in zip(image_item_ids, image_embeddings)
}

print("Image embedding map size:", len(image_emb_map))

In [ ]:
from numpy.linalg import norm

for k in image_emb_map:
    v = image_emb_map[k]
    image_emb_map[k] = v / norm(v)

In [ ]:
def image_similarity(item_i, item_j):

    emb_i = image_emb_map.get(item_i)
    emb_j = image_emb_map.get(item_j)

    if emb_i is None or emb_j is None:
     return np.nan

    return float(np.dot(emb_i, emb_j))



In [ ]:
from tqdm import tqdm

tqdm.pandas()

candidates["image_similarity"] = candidates.progress_apply(
    lambda x: image_similarity(
        df.loc[x["item_i"], "item_id"],
        df.loc[x["item_j"], "item_id"]
    ),
    axis=1
)

In [ ]:
candidates["image_similarity"] = (
    candidates["image_similarity"]
    .fillna(-1)
)

In [ ]:
candidates["image_similarity"].describe()

In [ ]:
candidates.to_parquet(
    "candidate_feature_table_multimodal.parquet",
    index=False
)

In [ ]:
import re

# Spec tokens that signal a VARIANT rather than a duplicate.
# (Validate this set against your 858 misclassified pairs before finalizing.)
SPEC_WORDS = {
    "gb", "tb", "mb",
    "mah",
    "hz", "ghz", "mhz",
    "inch", "cm", "mm", "ft",
    "oz", "ml", "l",
    "kg", "g", "lb",
    "pack", "packs",
    "ct", "count",
    "pc", "pcs",
    "piece", "pieces",
    "yr", "year", "years",
    "black", "white", "red", "blue", "green",
    "silver", "gold", "gray", "grey",
    "small", "medium", "large",
    "xs", "s", "m", "l", "xl", "xxl",
}
# NOTE: removed the preposition "in" (collides with real words) and the
# literal "2yr"/"3yr" (now redundant after the tokenizer split below).

NUM = re.compile(r"^\d+(\.\d+)?$")

def tokenize(title):
    t = re.sub(r"[^a-z0-9]", " ", str(title).lower())
    # FIX: split fused number+unit tokens -> "64gb"->"64 gb", "5yr"->"5 yr".
    # Without this, only the hardcoded 2yr/3yr were caught and every other
    # warranty term / capacity slipped through as a "duplicate".
    t = re.sub(r"(\d+)\s*([a-z]+)", r"\1 \2", t)
    return t.split()

def spec_difference(title1, title2):
    diff = set(tokenize(title1)) ^ set(tokenize(title2))
    if len(diff) == 0:
        return False
    return all(NUM.match(x) or x in SPEC_WORDS for x in diff)


In [ ]:
candidates["spec_diff"] = candidates.apply(
    lambda x: spec_difference(
        df.loc[x.item_i,"title"],
        df.loc[x.item_j,"title"]
    ),
    axis=1
)

In [ ]:
def weak_label(row):

    if (
        row.similarity > 0.92
        and row.image_similarity > 0.90
        and row.brand_match == 1
    ):
        if row.spec_diff:
            return "variant", 0.85
        return "duplicate", 0.95

    if (
        row.similarity > 0.80
        and row.brand_match == 1
        and row.category_match == 1
    ):
        return "variant", 0.80

    if row.similarity < 0.45:
        return "unrelated", 0.90

    return "unknown", 0.0


In [ ]:
candidates[["weak_label", "confidence"]] = candidates.apply(
    weak_label, axis=1, result_type="expand"
)


In [ ]:
candidates["weak_label"].value_counts()

In [ ]:
pair_edges = []

for _, row in candidates.iterrows():

    if row["weak_label"] == "unknown":
        continue

    item_i = f"item_{df.loc[row['item_i'],'item_id']}"
    item_j = f"item_{df.loc[row['item_j'],'item_id']}"

    relation = row["weak_label"]
    conf = float(row["confidence"])

    pair_edges.append((item_i, relation, item_j, conf))

    if relation in ["duplicate", "variant"]:
        pair_edges.append((item_j, relation, item_i, conf))

print("Total item-item edges:", len(pair_edges))


In [ ]:
import pandas as pd

pair_edges_df = pd.DataFrame(
    pair_edges,
    columns=["head", "relation", "tail", "confidence"]
)

pair_edges_df.to_parquet(
    "item_item_edges.parquet",
    index=False
)


In [ ]:
import pandas as pd

taxonomy_edges = pd.read_parquet("taxonomy_edges.parquet")
item_category_edges = pd.read_parquet("item_category_edges.parquet")
item_attribute_edges = pd.read_parquet("item_attribute_edges.parquet")
item_item_edges = pd.read_parquet("item_item_edges.parquet")

print("taxonomy:", len(taxonomy_edges))
print("item-category:", len(item_category_edges))
print("item-attribute:", len(item_attribute_edges))
print("item-item:", len(item_item_edges))

In [ ]:
import pandas as pd

taxonomy = pd.read_parquet("taxonomy_edges.parquet")
item_category = pd.read_parquet("item_category_edges.parquet")
item_attribute = pd.read_parquet("item_attribute_edges.parquet")
item_brand = pd.read_parquet("item_brand_edges_multimodal.parquet")
item_item = pd.read_parquet("item_item_edges.parquet")

# FIX: deterministic structural/attribute edges are confidence 1.0;
# item-item edges already carry their weak-label confidence.
for d in [taxonomy, item_category, item_attribute, item_brand]:
    d["confidence"] = 1.0

kg = pd.concat([
    taxonomy,
    item_category,
    item_attribute,
    item_brand,
    item_item
], ignore_index=True)

print("Total edges:", len(kg))
print("Unique relations:", kg["relation"].nunique())
print("Unique entities:",
      len(set(kg["head"]).union(set(kg["tail"]))))

kg.head()


In [ ]:
kg.to_parquet("product_knowledge_graph.parquet", index=False)

print("Saved final KG")

In [ ]:
entities = pd.unique(
    pd.concat([kg["head"], kg["tail"]], ignore_index=True)
)

entity2id = {e: i for i, e in enumerate(entities)}

print("Total entities:", len(entity2id))

In [ ]:
relations = kg["relation"].unique()

relation2id = {r: i for i, r in enumerate(relations)}

print("Total relations:", len(relation2id))

In [ ]:
kg['relation'].value_counts()

In [ ]:
kg_ids = pd.DataFrame({
    "head": kg["head"].map(entity2id),
    "relation": kg["relation"].map(relation2id),
    "tail": kg["tail"].map(entity2id),
    "confidence": kg["confidence"].values,
})

kg_ids.head()


In [ ]:
kg_ids.to_parquet("kg_triples_ids.parquet", index=False)

print("Saved KG triples with IDs")

In [ ]:
import pickle

pickle.dump(entity2id, open("entity2id.pkl","wb"))
pickle.dump(relation2id, open("relation2id.pkl","wb"))

In [ ]:
import json

kg_stats = {}

# Basic statistics
kg_stats["num_edges"] = len(kg)
kg_stats["num_relations"] = kg["relation"].nunique()

entities = set(kg["head"]).union(set(kg["tail"]))
kg_stats["num_entities"] = len(entities)

# Relation distribution
relation_counts = kg["relation"].value_counts().to_dict()
kg_stats["relation_distribution"] = relation_counts

# Node degree statistics
head_counts = kg["head"].value_counts()
tail_counts = kg["tail"].value_counts()

kg_stats["avg_out_degree"] = float(head_counts.mean())
kg_stats["avg_in_degree"] = float(tail_counts.mean())

kg_stats["max_out_degree"] = int(head_counts.max())
kg_stats["max_in_degree"] = int(tail_counts.max())

print("KG Statistics:")
for k,v in kg_stats.items():
    if k != "relation_distribution":
        print(k, ":", v)

In [ ]:
with open("kg_statistics.json", "w") as f:
    json.dump(kg_stats, f, indent=4)

print("KG statistics saved to kg_statistics.json")

In [ ]:
relation_df = kg["relation"].value_counts().reset_index()
relation_df.columns = ["relation", "count"]

relation_df.to_csv("kg_relation_distribution.csv", index=False)

relation_df.head()

In [ ]:
import networkx as nx

print("Building graph...")

G = nx.from_pandas_edgelist(
    kg,
    source="head",
    target="tail"
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

In [ ]:
components = list(nx.connected_components(G))

num_components = len(components)

largest_component_size = max(len(c) for c in components)

print("Number of components:", num_components)
print("Largest component size:", largest_component_size)

print("Percentage in giant component:",
      largest_component_size / G.number_of_nodes())

In [ ]:
component_sizes = [len(c) for c in components]

import pandas as pd

component_df = pd.DataFrame({
    "component_size": component_sizes
})

component_df.to_csv("kg_component_sizes.csv", index=False)

print("Saved component sizes.")

In [ ]:
degrees = [d for n, d in G.degree()]

degree_df = pd.DataFrame({"degree": degrees})

degree_df.to_csv("kg_degree_distribution.csv", index=False)

print("Saved degree distribution.")

In [ ]:
print("Average degree:", sum(degrees)/len(degrees))
print("Max degree:", max(degrees))
print("Median degree:", degree_df["degree"].median())

In [ ]:
import numpy as np

component_sizes = [len(c) for c in components]

print("Median component size:", np.median(component_sizes))
print("Max component size:", max(component_sizes))
print("Components size < 10:", sum(1 for x in component_sizes if x < 10))

In [ ]:
import pandas as pd

df = pd.read_parquet("candidate_feature_table_multimodal.parquet")

print(df.columns.tolist())

In [ ]:
import pandas as pd

pairs = pd.read_parquet("candidate_pairs.parquet")

print(pairs.columns.tolist())

In [ ]:
import pandas as pd

kg = pd.read_parquet("candidate_feature_table.parquet")

print(kg.columns.tolist())


In [ ]:
import pandas as pd

kg = pd.read_parquet("product_knowledge_graph.parquet")

print(kg.columns.tolist())
print(kg.head())

In [ ]:
print(kg["relation"].value_counts())

In [ ]:
import pandas as pd

# Load original product data
df = pd.read_parquet("merged_products.parquet")   # <-- replace with your actual merged dataset filename if different

title_map = dict(zip(df["item_id"], df["title"]))

kg = pd.read_parquet("product_knowledge_graph.parquet")

print("===== DUPLICATES =====")
dup = kg[kg["relation"]=="duplicate"].sample(10, random_state=42)

for _, r in dup.iterrows():
    print("="*80)
    print(r["head"], ":", title_map.get(r["head"], "NA"))
    print(r["tail"], ":", title_map.get(r["tail"], "NA"))

print("\n\n===== VARIANTS =====")
var = kg[kg["relation"]=="variant"].sample(10, random_state=42)

for _, r in var.iterrows():
    print("="*80)
    print(r["head"], ":", title_map.get(r["head"], "NA"))
    print(r["tail"], ":", title_map.get(r["tail"], "NA"))

# SANITY CHECKS — run after the full notebook to confirm the fixes

In [ ]:
import pandas as pd, pickle

kg_ids = pd.read_parquet("kg_triples_ids.parquet")
entity2id = pickle.load(open("entity2id.pkl", "rb"))
relation2id = pickle.load(open("relation2id.pkl", "rb"))
id2rel = {v: k for k, v in relation2id.items()}

n_ent = len(entity2id)
print("Entities:", n_ent, " (expect ~617K, NOT ~1.04M)")
assert n_ent < 750_000, (
    "GHOST NODES STILL PRESENT -> check item_ prefix on belongs_to / has_brand"
)

print("confidence column present:", "confidence" in kg_ids.columns)
assert "confidence" in kg_ids.columns, "Missing confidence column"

# bare-ASIN ghost items would appear as entities not starting with item_/attr_
ghosts = [e for e in entity2id
          if not str(e).startswith(("item_", "attr_")) and str(e)[:2] == "B0"]
print("Suspected bare-ASIN ghost nodes:", len(ghosts), "(expect 0)")

print("\nRelation distribution (duplicate should be << variant):")
print(kg_ids["relation"].map(id2rel).value_counts())
